Ex.1: Factorizarea Cholesky

In [4]:
import numpy as np
from scipy.linalg import lu_solve, lu_factor

def solve_cholesky_ldlt():
    # 1. initializare date
    n = 5  # ex: dimensiunea matricii
    eps = 1e-10

    # generam o matrice simetrica si pozitiv definita: A = B * B^T
    B = np.random.rand(n, n)
    A_init = np.dot(B, B.T)
    b = np.random.rand(n)

    # pastram o copie a matricii initiale  pentru verificarea finala
    A = A_init.copy()

    # 1): solutia din biblioteca (LU)
    lu, piv = lu_factor(A)
    x_lib = lu_solve((lu, piv), b)

    # 2) + restrictii: Descompunerea LDL^T (in-place)
    # reinitializam A cu valorile originale pentru Cholesky
    A = A_init.copy()
    d = np.zeros(n)

    for i in range(n):
        # calculam d[i]
        sum_d = 0
        for j in range(i):
            sum_d += (A[i, j] ** 2) * d[j] # calc contributia elementelor deja procesate

        d[i] = A[i, i] - sum_d

        # verificare la impartirea cu zero folosind ε
        if abs(d[i]) <= eps:
            print("Matricea nu este pozitiv definită sau este singulară.")
            return

        # calculam elementele L (stocate sub diagonala lui A)
        for i_alt in range(i + 1, n):
            sum_l = 0
            for j in range(i):
                sum_l += A[i_alt, j] * A[i, j] * d[j]
            A[i_alt, i] = (A[i_alt, i] - sum_l) / d[i] # aici suprascriu elementul original din A cu noua valoare din L

    # 3): Determinantul
    # det(A) = det(L) * det(D) * det(L^T).
    # Deoarece L are 1 pe diagonală, det(L) = 1. Deci det(A) = produsul elementelor din d
    det_A = np.prod(d)

    # 4): Rezolvarea Ax = b (Lz = b, Dy = z, L^T x = y)

    # Pas 1: Lz = b (substitutie directa, L are 1 pe diagonala)
    z = np.zeros(n)
    for i in range(n):
        sum_z = 0
        for j in range(i):
            sum_z += A[i, j] * z[j] # L[i,j] este stocat în A[i,j]
        z[i] = b[i] - sum_z

    # Pas 2: Dy = z
    y = z / d

    # Pas 3: L^T x = y (substitutie)
    x_chol = np.zeros(n)
    for i in range(n - 1, -1, -1):
        sum_x = 0
        for j in range(i + 1, n):
            sum_x += A[j, i] * x_chol[j] # L^T[i,j] este L[j,i], care e în A[j,i]
        x_chol[i] = y[i] - sum_x

    # 5): Verificare norme
    # implementare inmultire matrice-vector conform restrictiei
    Ax_chol = np.zeros(n)
    for i in range(n):
        for j in range(n):
            Ax_chol[i] += A_init[i, j] * x_chol[j]

    norm1 = np.linalg.norm(Ax_chol - b, 2)
    norm2 = np.linalg.norm(x_chol - x_lib, 2)

    # afisam rezultatele
    print(f"Determinant calculat: {det_A}")
    print(f"Norma ||A*x_chol - b||: {norm1:.12e}")
    print(f"Norma ||x_chol - x_lib||: {norm2:.12e}")

solve_cholesky_ldlt()

Determinant calculat: 0.09243263979004714
Norma ||A*x_chol - b||: 4.710277376051e-16
Norma ||x_chol - x_lib||: 5.857939004216e-15
